In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

FEATURE_PATH = "data/processed/CA_1_features_v1.parquet"

In [3]:
df = con.execute(f"""
SELECT
    item_id,
    dept_id,
    cat_id,
    store_id,
    state_id,
    date,
    units_sold,
    sell_price,
    weekday,
    month,
    year,
    event_name_1,
    event_type_1,
    snap_CA,
    lag_1,
    lag_7,
    lag_28,
    rolling_mean_7,
    rolling_mean_28
FROM read_parquet('{FEATURE_PATH}')
WHERE
    is_available = 1
    AND lag_28 IS NOT NULL
""").fetchdf()

print(df.shape)

(4751349, 19)


In [4]:
features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sell_price",
    "weekday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

In [5]:
con.execute(f"""
DESCRIBE SELECT *
FROM read_parquet('{FEATURE_PATH}')
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,item_id,VARCHAR,YES,None,None,None
1,dept_id,VARCHAR,YES,None,None,None
2,cat_id,VARCHAR,YES,None,None,None
3,store_id,VARCHAR,YES,None,None,None
4,state_id,VARCHAR,YES,None,None,None
5,d,VARCHAR,YES,None,None,None
6,units_sold,BIGINT,YES,None,None,None
7,date,TIMESTAMP_NS,YES,None,None,None
8,wm_yr_wk,BIGINT,YES,None,None,None
9,weekday,VARCHAR,YES,None,None,None


In [6]:
max_date = df["date"].max()

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

train = df[df["date"] < val_start].copy()

val = df[
    (df["date"] >= val_start) &
    (df["date"] < test_start)
].copy()

test = df[df["date"] >= test_start].copy()

print(train.shape, val.shape, test.shape)

(4580605, 19) (85372, 19) (85372, 19)


In [7]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 28.8 MB/s eta 0:00:0031m30.0 MB/s eta 0:00:01


In [11]:
features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sell_price",
    "weekday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

In [12]:
print(train.columns.tolist())

missing_features = [
    col for col in features
    if col not in train.columns
]

print("Missing features:", missing_features)

['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'date', 'units_sold', 'sell_price', 'weekday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']
Missing features: []


In [13]:
cat_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1"
]

for col in cat_features:
    train[col] = train[col].fillna("None").astype(str)
    val[col] = val[col].fillna("None").astype(str)
    test[col] = test[col].fillna("None").astype(str)

In [14]:
model.fit(
    train[features],
    train[target],
    cat_features=cat_features,
    eval_set=(val[features], val[target]),
    early_stopping_rounds=50
)

0:	learn: 4.3172432	test: 3.3670157	best: 3.3670157 (0)	total: 1.15s	remaining: 9m 32s
100:	learn: 2.3958775	test: 2.0361921	best: 2.0361825 (98)	total: 1m 44s	remaining: 6m 52s
200:	learn: 2.3563111	test: 2.0250602	best: 2.0250602 (200)	total: 3m 33s	remaining: 5m 18s
300:	learn: 2.3333528	test: 2.0193266	best: 2.0193266 (300)	total: 5m 29s	remaining: 3m 38s
400:	learn: 2.3141563	test: 2.0158084	best: 2.0158080 (399)	total: 7m 27s	remaining: 1m 50s
499:	learn: 2.2990139	test: 2.0129142	best: 2.0128242 (471)	total: 9m 35s	remaining: 0us

bestTest = 2.012824153
bestIteration = 471

Shrink model to first 472 iterations.


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [15]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred = model.predict(val[features])

mae = mean_absolute_error(
    val[target],
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        val[target],
        pred
    )
)

print("CatBoost MAE:", mae)
print("CatBoost RMSE:", rmse)

CatBoost MAE: 1.0384569631641956
CatBoost RMSE: 2.0128241533877698


In [18]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

model.save_model("models/catboost_ca1.cbm")